<a href="https://colab.research.google.com/github/oluwayemidara1/data-portfolio/blob/main/comfyui_ltx25main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub

# Cloudflare tunnel so you can open the UI in your browser
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Optional: official LTX nodes (extra example workflows, IC-LoRA, etc.)
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git
!pip install -q -r ComfyUI-LTXVideo/requirements.txt
%cd /content/ComfyUI

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 61285, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 61285 (delta 5), reused 4 (delta 2), pack-reused 61272 (from 2)
Receiving objects: 100% (61285/61285), 102.44 MiB | 23.95 MiB/s, done.
Resolving deltas: 100% (42551/42551), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 MB 9.8 MB/s

In [ ]:
!pip install -q "huggingface_hub>=1.32,<2.0"

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# "int8" = smaller, fits 24GB cards (L4). "bf16" = full precision, for A100/H100.
VARIANT = "int8"   # or "bf16"

common = [
    "vae/ltx-2.5-video-vae-bf16.safetensors",
    "vae/ltx-2.5-audio-vae-bf16.safetensors",
    "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
    "model_patches/ltx-2.5-duration-head-bf16.safetensors",
]

if VARIANT == "int8":
    files = common + [
        "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
    ]
else:
    files = common + [
        "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors",
        "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors",
    ]

# Uncomment for the full (non-distilled) dev model + distilled LoRA:
# files += [
#     "diffusion_models/ltx-2.5-22b-dev-transformer-comfy-int8-convrot.safetensors",  # or the bf16 dev file
#     "loras/ltx-2.5-22b-distilled-lora-450-bf16.safetensors",
# ]

file_args = " ".join(files)
!hf download Lightricks/LTX-2.5 {file_args} --local-dir /content/ComfyUI/models

Hint: A new version of huggingface_hub (2.2.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/365M [00:00<?, ?B/s]          
Reconstructing (incomplete total...):   0% 0.00/21.9G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/39.7G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/39.7G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/39.7G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 3.84M/39.7G [00:00<1:53:05, 5.85MB/s]

Reconstructing (incomplete total...):   1% 205M/39.7G [00:05<20:56, 31.4MB/s, 87.4MB/s  ]
Reconstructing (incomplete total...):   1% 419M/39.7G [00:05<06:57, 94.2MB/s, 31.4MB/s  ]
Reconstructing (incompl

In [ ]:
import subprocess, time, re, socket, os

# Kill any leftovers from a previous attempt
!pkill -f "main.py"; pkill -f cloudflared; true
time.sleep(2)

# Start ComfyUI, logging to a file
log = open("/content/comfy.log", "w")
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)

# Wait until the port actually accepts connections (up to 5 min)
def port_open():
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", 8188)) == 0

for i in range(150):
    if comfy.poll() is not None:
        print("ComfyUI crashed. Last log lines:\n")
        print("".join(open("/content/comfy.log").readlines()[-40:]))
        raise SystemExit
    if port_open():
        print(f"ComfyUI is up after ~{i*2}s")
        break
    time.sleep(2)
else:
    print("Timed out. Last log lines:\n")
    print("".join(open("/content/comfy.log").readlines()[-40:]))
    raise SystemExit

# Only now open the tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Open ComfyUI here:", m.group(0))
        break

# Keep the cell alive and report if ComfyUI dies
while True:
    time.sleep(30)
    if comfy.poll() is not None:
        print("ComfyUI stopped. Last log lines:\n")
        print("".join(open("/content/comfy.log").readlines()[-40:]))
        break

^C
ComfyUI crashed. Last log lines:

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[WARNING] Could not autodetect AIMDO implementation, assuming Nvidia
[INFO] Found comfy_kitchen backend triton: {'available': False, 'disabled': True, 'unavailable_reason': "ImportError: No module named 'triton'", 'capabilities': []}
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': False, 'disabled': True, 'unavailable_reason': 'CUDA not available on this system', 'capabilities': []}
[INFO] Found comfy_kitchen backend eager: {'available'

SystemExit: 

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
